# MUFASA Research Evidence Reranker — MiniLM-L6

This notebook follows `Otto-Destiny/assetops-minilm-l6-reranker` with the same base model, `cross-encoder/ms-marco-MiniLM-L6-v2`, but uses the paper JSON corpus in Google Drive.

The corpus is scanned **recursively**, including every nested subfolder. One shared `paper_id` manifest assigns **80% train / 10% validation / 10% hidden test**. The LFM notebooks use this exact same manifest, so a source paper can never be train in one model and validation/test in another.

The hidden test is not passed to Trainer, not used for checkpoint selection, and remains gated until all train/validation decisions are finished.

In [1]:
#@title 1. Install dependencies — do NOT upgrade Colab's NumPy/SciPy stack
%pip install -q transformers datasets accelerate kaggle

In [2]:
#@title 2. Imports and runtime check

import os
import re
import json
import math
import time
import random
import shutil
import inspect
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print("VRAM (GB):", round(props.total_memory / 1024**3, 2))
else:
    print("CPU training is supported, but a GPU will be much faster.")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM (GB): 39.49


In [3]:
#@title 3. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
#@title 4. Configuration — same MiniLM recipe + shared train/val/test split

JSON_DIR = "/content/drive/MyDrive/ADTC/data"  #@param {type:"string"}
OUTPUT_ROOT = "/content/drive/MyDrive/ADTC/mufasa_minilm_l6_research_reranker"  #@param {type:"string"}
BASE_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"
MAX_LENGTH = 256
SEED = 42
NUM_EPOCHS = 2.0
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
ADD_RANDOM_NEGATIVES = False
RANDOM_NEGATIVES_PER_QUERY = 0
KAGGLE_OWNER = "williamalabi"  #@param {type:"string"}
KAGGLE_DATASET_TITLE = "mufasa-minilm-l6-research-reranker"  #@param {type:"string"}
KAGGLE_DATASET_SLUG = f"{KAGGLE_OWNER}/{KAGGLE_DATASET_TITLE}"
os.makedirs(OUTPUT_ROOT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [5]:
#@title 5. FAST recursive discovery + SHARED train/val/hidden-test split
import os, random
from pathlib import Path
from collections import Counter
import pandas as pd

SPLIT_MANIFEST_PATH="/content/drive/MyDrive/ADTC/data/mufasa_paper_split_manifest.csv"
TRAIN_FRACTION,VAL_FRACTION,TEST_FRACTION=0.80,0.10,0.10
SPLIT_SEED=42
REBUILD_SPLIT_MANIFEST=False
RUN_HIDDEN_TEST=False

def discover_json_files_fast(json_root):
    root=Path(json_root).expanduser()
    if not Path("/content/drive/MyDrive").exists(): raise FileNotFoundError("Google Drive is not mounted.")
    if not root.exists(): raise FileNotFoundError(f"Dataset root does not exist: {root}")
    paths=[]
    for current_dir,dirnames,filenames in os.walk(root,topdown=True,followlinks=True):
        dirnames.sort(); filenames.sort()
        current_dir=Path(current_dir)
        for filename in filenames:
            if filename.lower().endswith(".json") and filename.lower() not in {"dataset-metadata.json","mufasa_paper_split_manifest.json"}:
                paths.append(current_dir/filename)
    paths=sorted(set(paths))
    if not paths: raise FileNotFoundError(f"No JSON files found recursively under {root}")
    counts=Counter((p.relative_to(root).parts[0] if len(p.relative_to(root).parts)>1 else "ROOT") for p in paths)
    print(f"Found {len(paths):,} JSON files.")
    print("By folder:",dict(sorted(counts.items())))
    return paths

def build_file_index(json_root):
    root=Path(json_root); paths=discover_json_files_fast(root)
    df=pd.DataFrame([{"paper_id":p.stem,"source_path":str(p),"relative_path":str(p.relative_to(root))} for p in paths])
    dupes=df[df.duplicated("paper_id",keep=False)].copy()
    papers=df.sort_values(["paper_id","relative_path"]).drop_duplicates("paper_id",keep="first").reset_index(drop=True)
    return papers,dupes,len(paths)

def make_split_map(paper_ids):
    ids=sorted(set(map(str,paper_ids))); n=len(ids)
    if n<3: raise ValueError(f"Need at least 3 papers; found {n}")
    rng=random.Random(SPLIT_SEED); rng.shuffle(ids)
    n_test=max(1,round(n*TEST_FRACTION)); n_val=max(1,round(n*VAL_FRACTION))
    while n_test+n_val>=n:
        if n_test>=n_val and n_test>1: n_test-=1
        elif n_val>1: n_val-=1
        else: break
    test=set(ids[:n_test]); val=set(ids[n_test:n_test+n_val])
    return {pid:"test" if pid in test else "val" if pid in val else "train" for pid in ids}

def build_or_load_shared_manifest(json_root):
    path=Path(SPLIT_MANIFEST_PATH)
    if path.exists() and not REBUILD_SPLIT_MANIFEST:
        manifest=pd.read_csv(path,dtype={"paper_id":str})
        required={"paper_id","source_path","relative_path","split"}
        missing=required-set(manifest.columns)
        if missing: raise RuntimeError(f"Invalid manifest, missing: {missing}. Set REBUILD_SPLIT_MANIFEST=True once.")
        if manifest.paper_id.duplicated().any(): raise RuntimeError("Duplicate paper_id in manifest.")
        print(f"Loaded manifest: {path}")
        print(f"Unique papers: {len(manifest):,}")
        print(manifest.split.value_counts().to_string())
        return manifest,pd.DataFrame(),pd.DataFrame()

    papers,dupes,physical_count=build_file_index(json_root)
    papers["split"]=papers.paper_id.map(make_split_map(papers.paper_id))
    path.parent.mkdir(parents=True,exist_ok=True)
    papers.to_csv(path,index=False)
    print(f"Physical JSON files: {physical_count:,}")
    print(f"Unique papers: {len(papers):,}")
    print(f"Duplicate copies: {len(dupes):,}")
    print(papers.split.value_counts().to_string())
    return papers,dupes,pd.DataFrame()

paper_manifest,duplicate_paper_files,bad_json_files=build_or_load_shared_manifest(JSON_DIR)
PAPER_SPLIT=dict(zip(paper_manifest.paper_id.astype(str),paper_manifest.split))
print(f"Manifest ready: {len(PAPER_SPLIT):,} papers")

Loaded manifest: /content/drive/MyDrive/ADTC/data/mufasa_paper_split_manifest.csv
Unique papers: 4,986
split
train    3988
test      499
val       499
Manifest ready: 4,986 papers


## Build grouped relevance supervision

The hard negatives in your JSON are especially valuable because they are not random unrelated passages. They are deliberately confusing alternatives such as:

- same paper, other section,
- same material, other property,
- superficial term overlap.

We therefore keep these supplied hard negatives as the default training negatives rather than diluting them with easy random passages.

In [6]:
#@title 6. FAST parallel reranker parsing + cache
import json,re,os
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

RERANKER_CACHE="/content/drive/MyDrive/ADTC/data/mufasa_reranker_groups.pkl"
REBUILD_RERANKER_CACHE=False
IO_WORKERS=24

def clean_text(x):
    if x is None:return ""
    x=str(x).replace("\u00a0"," "); x=re.sub(r"[ \t]+"," ",x); return re.sub(r"\n{3,}","\n\n",x).strip()

def as_list(x):
    if x is None:return []
    if isinstance(x,list):return [clean_text(v) for v in x if clean_text(v)]
    x=clean_text(x); return [x] if x else []

def parse_one(row):
    path=Path(row.source_path); paper_id=str(row.paper_id); split=str(row.split)
    try:
        with open(path,"r",encoding="utf-8") as f:data=json.load(f)
        title=clean_text(data.get("title")); domain=clean_text(data.get("manifest_domain")) or "UNKNOWN"
        rr=data.get("tasks",{}).get("training",{}).get("payload",{}).get("reranker",[]) or []
        out=[]; skipped=[]
        for i,item in enumerate(rr):
            q=clean_text(item.get("query")); pos=as_list(item.get("positive_quote")); neg=as_list(item.get("hard_negative_quote"))
            lid=clean_text(item.get("local_id")) or f"reranker_{i:03d}"; reason=clean_text(item.get("negative_reason")) or "UNKNOWN"
            if not pos:
                fb=clean_text((item.get("evidence") or {}).get("quote"))
                if fb:pos=[fb]
            pset={x.casefold() for x in pos}; neg=[x for x in neg if x.casefold() not in pset]
            if not q or not pos or not neg:
                skipped.append({"paper_id":paper_id,"local_id":lid,"reason":"missing query/positive/hard-negative"}); continue
            for p in pos:
                for n in neg:
                    out.append({"group_id":f"{paper_id}::{lid}","paper_id":paper_id,"split":split,"domain":domain,"local_id":lid,"query":q,"positive_quote":p,"hard_negative_quote":n,"negative_reason":reason,"source_file":str(path)})
        return out,skipped
    except Exception as e:
        return [],[{"paper_id":paper_id,"local_id":"","reason":repr(e)}]

cache=Path(RERANKER_CACHE)

if cache.exists() and not REBUILD_RERANKER_CACHE:
    groups_df=pd.read_pickle(cache)
    skipped_rerankers=[]
    print(f"Loaded cached reranker data: {len(groups_df):,} groups")
else:
    rows=list(paper_manifest.itertuples(index=False))
    print(f"Parsing {len(rows):,} papers with {IO_WORKERS} parallel readers...")
    all_rows=[]; skipped_rerankers=[]
    with ThreadPoolExecutor(max_workers=IO_WORKERS) as ex:
        for out,skipped in ex.map(parse_one,rows,chunksize=16):
            all_rows.extend(out); skipped_rerankers.extend(skipped)
    groups_df=pd.DataFrame(all_rows)
    if groups_df.empty:raise RuntimeError("No usable reranker groups extracted.")
    groups_df.to_pickle(cache)
    print(f"Saved cache: {cache}")

all_groups=[{
    "group_id":r.group_id,"paper_id":r.paper_id,"split":r.split,"domain":r.domain,"local_id":r.local_id,
    "query":r.query,"docs":[r.positive_quote,r.hard_negative_quote],"labels":[1,0],
    "negative_reason":r.negative_reason,"source_file":r.source_file
} for r in groups_df.itertuples(index=False)]

print(f"Reranker groups: {len(groups_df):,}")
print(f"Papers with rerankers: {groups_df.paper_id.nunique():,}")
print(f"Skipped: {len(skipped_rerankers):,}")
print("\nGroups by split:\n"+groups_df.split.value_counts().to_string())
print("\nNegative reasons:\n"+groups_df.negative_reason.value_counts().to_string())

Loaded cached reranker data: 20,671 groups
Reranker groups: 20,671
Papers with rerankers: 4,151
Skipped: 0

Groups by split:
split
train    16503
test      2105
val       2063

Negative reasons:
negative_reason
SAME_MATERIAL_OTHER_PROPERTY                                                                                                                                                                            6897
SAME_PAPER_OTHER_SECTION                                                                                                                                                                                6489
SAME_PROPERTY_OTHER_MATERIAL                                                                                                                                                                            3970
SUPERFICIAL_TERM_OVERLAP                                                                                                                                                      

In [7]:
#@title 7. Apply shared TRAIN / VALIDATION / HIDDEN TEST paper split
train_groups=[g for g in all_groups if g["split"]=="train"]
val_groups=[g for g in all_groups if g["split"]=="val"]
test_groups=[g for g in all_groups if g["split"]=="test"]
assert train_groups and val_groups and test_groups
train_papers={g["paper_id"] for g in train_groups}; val_papers={g["paper_id"] for g in val_groups}; test_papers={g["paper_id"] for g in test_groups}
assert train_papers.isdisjoint(val_papers) and train_papers.isdisjoint(test_papers) and val_papers.isdisjoint(test_papers)
print("Train groups:",len(train_groups),"papers:",len(train_papers))
print("Validation groups:",len(val_groups),"papers:",len(val_papers))
print("Hidden-test groups:",len(test_groups),"papers:",len(test_papers))
print("Hidden test locked:",not RUN_HIDDEN_TEST)

Train groups: 16503 papers: 3317
Validation groups: 2063 papers: 413
Hidden-test groups: 2105 papers: 421
Hidden test locked: True


In [8]:
#@title 8. Convert ranking groups into binary query-candidate training pairs

def groups_to_pairs(groups):
    pairs = []
    for group in groups:
        for doc, label in zip(group["docs"], group["labels"]):
            pairs.append({
                "query": group["query"],
                "doc": doc,
                "label": float(label),
                "group_id": group["group_id"],
                "paper_id": group["paper_id"],
                "domain": group["domain"],
                "negative_reason": group["negative_reason"],
            })
    return pairs

train_pairs = groups_to_pairs(train_groups)
val_pairs = groups_to_pairs(val_groups)
test_pairs = groups_to_pairs(test_groups)

# Optional easy/random negatives are OFF by default.
if ADD_RANDOM_NEGATIVES and RANDOM_NEGATIVES_PER_QUERY > 0:
    negative_pool = [
        doc
        for group in train_groups
        for doc, label in zip(group["docs"], group["labels"])
        if label == 0
    ]
    rng = random.Random(SEED)

    existing = {
        (p["group_id"], p["doc"].casefold())
        for p in train_pairs
    }

    for group in train_groups:
        candidates = [
            d for d in negative_pool
            if (group["group_id"], d.casefold()) not in existing
        ]
        if candidates:
            sampled = rng.sample(
                candidates,
                min(RANDOM_NEGATIVES_PER_QUERY, len(candidates))
            )
            for doc in sampled:
                train_pairs.append({
                    "query": group["query"],
                    "doc": doc,
                    "label": 0.0,
                    "group_id": group["group_id"],
                    "paper_id": group["paper_id"],
                    "domain": group["domain"],
                    "negative_reason": "RANDOM_EXTRA_NEGATIVE",
                })

train_pairs_df = pd.DataFrame(train_pairs)
val_pairs_df = pd.DataFrame(val_pairs)

print("Train pairs:", len(train_pairs_df))
print("Validation pairs:", len(val_pairs_df))
print("Hidden-test pairs:", len(test_pairs))
print("\nTrain label balance:")
print(train_pairs_df.label.value_counts().sort_index().to_string())

# Exact pair leakage audit.
def norm_pair(q, d):
    q = re.sub(r"\s+", " ", q).strip().casefold()
    d = re.sub(r"\s+", " ", d).strip().casefold()
    return q, d

train_pair_keys = {
    norm_pair(q, d)
    for q, d in zip(train_pairs_df["query"], train_pairs_df["doc"])
}
cross_split_duplicates = sum(
    norm_pair(q, d) in train_pair_keys
    for q, d in zip(val_pairs_df["query"], val_pairs_df["doc"])
)

print("\nExact query+candidate duplicates crossing split:", cross_split_duplicates)

train_pairs_df.to_csv(
    os.path.join(OUTPUT_ROOT, "train_pairs.csv"), index=False
)
val_pairs_df.to_csv(
    os.path.join(OUTPUT_ROOT, "validation_pairs.csv"), index=False
)

with open(
    os.path.join(OUTPUT_ROOT, "validation_groups.jsonl"),
    "w",
    encoding="utf-8",
) as f:
    for g in val_groups:
        f.write(json.dumps(g, ensure_ascii=False) + "\n")

Train pairs: 33006
Validation pairs: 4126
Hidden-test pairs: 4210

Train label balance:
label
0.0    16503
1.0    16503

Exact query+candidate duplicates crossing split: 0


## Tokenization and loss

This follows the reference repository:

```python
tokenizer(query, candidate, truncation=True, max_length=256, padding=False)
```

Batch padding is performed later with `DataCollatorWithPadding`.

The classifier has one scalar logit per query-candidate pair. Training uses binary cross entropy on that logit.

In [9]:
#@title 8. Load tokenizer and create tokenized datasets

from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = (
        tokenizer.eos_token
        if tokenizer.eos_token is not None
        else tokenizer.sep_token
    )

def pairs_to_hf_dataset(pairs):
    frame = pd.DataFrame(pairs)
    dataset = Dataset.from_pandas(frame, preserve_index=False)

    def tokenize_batch(batch):
        return tokenizer(
            batch["query"],
            batch["doc"],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )

    remove_columns = [
        c for c in [
            "query", "doc", "group_id", "paper_id",
            "domain", "negative_reason"
        ]
        if c in dataset.column_names
    ]

    dataset = dataset.map(
        tokenize_batch,
        batched=True,
        remove_columns=remove_columns,
    )

    if "label" in dataset.column_names:
        dataset = dataset.rename_column("label", "labels")

    return dataset

train_dataset = pairs_to_hf_dataset(train_pairs)
eval_dataset = pairs_to_hf_dataset(val_pairs)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_dataset)
print(eval_dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Map:   0%|          | 0/33006 [00:00<?, ? examples/s]

Map:   0%|          | 0/4126 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 33006
})
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4126
})


In [10]:
#@title 9. Grouped ranking evaluation functions — same metric family as reference

from tqdm.auto import tqdm

METRIC_COLUMNS = [
    "top1_accuracy",
    "exact_match_at_k",
    "precision_at_k",
    "recall_at_k",
    "f1_at_k",
    "mrr",
    "ndcg_at_k",
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

@torch.inference_mode()
def score_query_docs(model, query, docs, batch_size=64):
    model.eval()
    model.to(DEVICE)

    scores = []

    for start in range(0, len(docs), batch_size):
        batch_docs = list(docs[start:start + batch_size])

        enc = tokenizer(
            [query] * len(batch_docs),
            batch_docs,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_tensors="pt",
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        logits = model(**enc).logits.squeeze(-1)
        probs = torch.sigmoid(logits).detach().cpu().numpy().tolist()

        if isinstance(probs, float):
            probs = [probs]

        scores.extend(probs)

    return scores

def dcg_at_k(relevances, k):
    rels = np.asarray(relevances, dtype=float)[:k]
    if rels.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, rels.size + 2))
    return float(np.sum(rels / discounts))

def ndcg_at_k(labels_ordered, k):
    dcg = dcg_at_k(labels_ordered, k)
    ideal = sorted(labels_ordered, reverse=True)
    idcg = dcg_at_k(ideal, k)
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_groups(model, groups, model_stage, batch_size=64):
    rows = []

    for group in tqdm(groups, desc=f"Evaluating {model_stage}"):
        labels = np.asarray(group["labels"], dtype=int)
        if labels.sum() == 0:
            continue

        scores = np.asarray(
            score_query_docs(
                model,
                group["query"],
                group["docs"],
                batch_size=batch_size,
            )
        )

        order = np.argsort(-scores)
        ordered_labels = labels[order]

        # Our schema normally contains one positive per group.
        k = max(1, min(int(labels.sum()), len(labels)))
        topk_labels = ordered_labels[:k]

        precision = float(topk_labels.sum() / k)
        recall = float(topk_labels.sum() / labels.sum())
        f1 = (
            float(2 * precision * recall / (precision + recall))
            if precision + recall > 0
            else 0.0
        )

        exact_match = float(
            topk_labels.sum() == labels.sum()
            and k == labels.sum()
        )
        top1_accuracy = float(ordered_labels[0] == 1)

        correct_ranks = np.where(ordered_labels == 1)[0]
        mrr = (
            float(1.0 / (correct_ranks[0] + 1))
            if len(correct_ranks)
            else 0.0
        )

        ndcg = ndcg_at_k(ordered_labels, k)

        rows.append({
            "model_stage": model_stage,
            "group_id": group["group_id"],
            "paper_id": group["paper_id"],
            "domain": group["domain"],
            "negative_reason": group["negative_reason"],
            "num_candidates": len(labels),
            "num_correct": int(labels.sum()),
            "k": k,
            "top1_accuracy": top1_accuracy,
            "exact_match_at_k": exact_match,
            "precision_at_k": precision,
            "recall_at_k": recall,
            "f1_at_k": f1,
            "mrr": mrr,
            "ndcg_at_k": ndcg,
        })

    detail = pd.DataFrame(rows)

    overall = (
        detail[METRIC_COLUMNS]
        .mean()
        .to_frame()
        .T
    )
    overall.insert(0, "model_stage", model_stage)

    by_domain = (
        detail.groupby("domain")[METRIC_COLUMNS]
        .mean()
        .reset_index()
    )
    by_domain.insert(0, "model_stage", model_stage)

    by_negative_reason = (
        detail.groupby("negative_reason")[METRIC_COLUMNS]
        .mean()
        .reset_index()
    )
    by_negative_reason.insert(0, "model_stage", model_stage)

    return detail, overall, by_domain, by_negative_reason

## Raw base-model baseline

The reference project emphasizes loading a **fresh raw base model** rather than accidentally evaluating the already fine-tuned object twice. We do the same here.

In [11]:
#@title 10. Evaluate the untouched raw MiniLM reranker

from transformers import AutoModelForSequenceClassification

raw_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=1,
    trust_remote_code=True,
    ignore_mismatched_sizes=True,
)

raw_detail, raw_overall, raw_by_domain, raw_by_reason = evaluate_groups(
    raw_model,
    val_groups,
    model_stage="raw_base",
    batch_size=64,
)

print("RAW BASE overall ranking metrics:")
display(raw_overall)

raw_detail.to_csv(
    os.path.join(OUTPUT_ROOT, "raw_base_group_metrics.csv"),
    index=False,
)
raw_by_domain.to_csv(
    os.path.join(OUTPUT_ROOT, "raw_base_by_domain.csv"),
    index=False,
)
raw_by_reason.to_csv(
    os.path.join(OUTPUT_ROOT, "raw_base_by_negative_reason.csv"),
    index=False,
)

# Free GPU memory before training a fresh model.
raw_model.to("cpu")
del raw_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Evaluating raw_base:   0%|          | 0/2063 [00:00<?, ?it/s]

RAW BASE overall ranking metrics:


,model_stage,top1_accuracy,exact_match_at_k,precision_at_k,recall_at_k,f1_at_k,mrr,ndcg_at_k
0,raw_base,0.833737,0.833737,0.833737,0.833737,0.833737,0.916869,0.833737


In [12]:
#@title 11. Load a fresh model and define BCEWithLogitsLoss Trainer

import torch.nn as nn
from transformers import Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=1,
    trust_remote_code=True,
    ignore_mismatched_sizes=True,
)

class BinaryRerankerTrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs,
    ):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits = outputs.logits.squeeze(-1)
        loss = nn.BCEWithLogitsLoss()(logits, labels)
        return (loss, outputs) if return_outputs else loss

steps_per_epoch = math.ceil(
    len(train_dataset) / TRAIN_BATCH_SIZE
)
warmup_steps = max(
    10,
    int(0.05 * steps_per_epoch * NUM_EPOCHS)
)

args_kwargs = dict(
    output_dir=os.path.join(OUTPUT_ROOT, "checkpoints"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
    seed=SEED,
)

ta_params = inspect.signature(
    TrainingArguments.__init__
).parameters

if "eval_strategy" in ta_params:
    args_kwargs["eval_strategy"] = "steps"
else:
    args_kwargs["evaluation_strategy"] = "steps"

training_args = TrainingArguments(**args_kwargs)

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

trainer_params = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = BinaryRerankerTrainer(**trainer_kwargs)

print("Train pairs:", len(train_dataset))
print("Eval pairs:", len(eval_dataset))
print("Warmup steps:", warmup_steps)
print("Device:", DEVICE)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Train pairs: 33006
Eval pairs: 4126
Warmup steps: 412
Device: cuda


In [13]:
#@title 12. Fine-tune MiniLM-L6 reranker

train_result = trainer.train()

print("\nTraining metrics:")
print(train_result.metrics)

MODEL_DIR = os.path.join(
    OUTPUT_ROOT,
    "mufasa_minilm_l6_research_reranker"
)
os.makedirs(MODEL_DIR, exist_ok=True)

trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

with open(
    os.path.join(OUTPUT_ROOT, "training_metrics.json"),
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            k: (
                float(v)
                if isinstance(v, (int, float, np.number))
                else str(v)
            )
            for k, v in train_result.metrics.items()
        },
        f,
        indent=2,
    )

print("Saved model to:", MODEL_DIR)

Step,Training Loss,Validation Loss
50,1.634159,1.345060
100,1.382586,1.201191
150,1.172954,1.006496
200,0.838802,0.783710
250,0.618242,0.687199
300,0.625632,0.601711
350,0.603639,0.566856
400,0.576652,0.575933
450,0.527213,0.601768
500,0.571831,0.571077


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training metrics:
{'train_runtime': 693.7202, 'train_samples_per_second': 95.157, 'train_steps_per_second': 11.895, 'total_flos': 401545773148116.0, 'train_loss': 0.48484525831359365, 'epoch': 2.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to: /content/drive/MyDrive/ADTC/mufasa_minilm_l6_research_reranker/mufasa_minilm_l6_research_reranker


In [14]:
#@title 13. Evaluate the fine-tuned reranker on the SAME held-out paper groups

fine_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
    num_labels=1,
    trust_remote_code=True,
)

fine_detail, fine_overall, fine_by_domain, fine_by_reason = evaluate_groups(
    fine_model,
    val_groups,
    model_stage="fine_tuned",
    batch_size=64,
)

print("FINE-TUNED overall ranking metrics:")
display(fine_overall)

fine_detail.to_csv(
    os.path.join(OUTPUT_ROOT, "fine_tuned_group_metrics.csv"),
    index=False,
)
fine_by_domain.to_csv(
    os.path.join(OUTPUT_ROOT, "fine_tuned_by_domain.csv"),
    index=False,
)
fine_by_reason.to_csv(
    os.path.join(OUTPUT_ROOT, "fine_tuned_by_negative_reason.csv"),
    index=False,
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Evaluating fine_tuned:   0%|          | 0/2063 [00:00<?, ?it/s]

FINE-TUNED overall ranking metrics:


,model_stage,top1_accuracy,exact_match_at_k,precision_at_k,recall_at_k,f1_at_k,mrr,ndcg_at_k
0,fine_tuned,0.888512,0.888512,0.888512,0.888512,0.888512,0.944256,0.888512


In [15]:
#@title 14. Raw vs fine-tuned comparison

comparison_rows = []

for metric in METRIC_COLUMNS:
    base_value = float(raw_overall.iloc[0][metric])
    fine_value = float(fine_overall.iloc[0][metric])

    comparison_rows.append({
        "metric": metric,
        "raw_base": base_value,
        "fine_tuned": fine_value,
        "absolute_improvement": fine_value - base_value,
        "relative_improvement_percent": (
            (fine_value - base_value) / base_value * 100.0
            if base_value != 0
            else None
        ),
    })

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df)

comparison_df.to_csv(
    os.path.join(OUTPUT_ROOT, "raw_vs_finetuned_metrics.csv"),
    index=False,
)

print("\nFine-tuned by domain:")
display(fine_by_domain)

print("\nFine-tuned by hard-negative type:")
display(fine_by_reason)

,metric,raw_base,fine_tuned,absolute_improvement,relative_improvement_percent
0,top1_accuracy,0.833737,0.888512,0.054775,6.569767
1,exact_match_at_k,0.833737,0.888512,0.054775,6.569767
2,precision_at_k,0.833737,0.888512,0.054775,6.569767
3,recall_at_k,0.833737,0.888512,0.054775,6.569767
4,f1_at_k,0.833737,0.888512,0.054775,6.569767
5,mrr,0.916869,0.944256,0.027387,2.987047
6,ndcg_at_k,0.833737,0.888512,0.054775,6.569767



Fine-tuned by domain:


,model_stage,domain,top1_accuracy,exact_match_at_k,precision_at_k,recall_at_k,f1_at_k,mrr,ndcg_at_k
0,fine_tuned,AGR,0.909492,0.909492,0.909492,0.909492,0.909492,0.954746,0.909492
1,fine_tuned,ENR,0.916129,0.916129,0.916129,0.916129,0.916129,0.958065,0.916129
2,fine_tuned,ENV,0.876518,0.876518,0.876518,0.876518,0.876518,0.938259,0.876518
3,fine_tuned,HLT,0.876559,0.876559,0.876559,0.876559,0.876559,0.938279,0.876559
4,fine_tuned,MAT,0.915966,0.915966,0.915966,0.915966,0.915966,0.957983,0.915966
5,fine_tuned,TEC,0.850000,0.850000,0.850000,0.850000,0.850000,0.925000,0.850000



Fine-tuned by hard-negative type:


,model_stage,negative_reason,top1_accuracy,exact_match_at_k,precision_at_k,recall_at_k,f1_at_k,mrr,ndcg_at_k
0,fine_tuned,SAME_MATERIAL_OTHER_PROPERTY,0.860741,0.860741,0.860741,0.860741,0.860741,0.930370,0.860741
1,fine_tuned,SAME_PAPER_OTHER_SECTION,0.905918,0.905918,0.905918,0.905918,0.905918,0.952959,0.905918
2,fine_tuned,SAME_PROPERTY_OTHER_MATERIAL,0.882353,0.882353,0.882353,0.882353,0.882353,0.941176,0.882353
3,fine_tuned,SUPERFICIAL_TERM_OVERLAP,0.919003,0.919003,0.919003,0.919003,0.919003,0.959502,0.919003


In [16]:
#@title 15. OPTIONAL FINAL hidden-test evaluation — locked until tuning is finished
if not RUN_HIDDEN_TEST:
    print("Hidden test remains LOCKED. Set RUN_HIDDEN_TEST=True only after all train/validation model decisions are final.")
else:
    hidden_detail,hidden_overall,hidden_by_domain,hidden_by_reason=evaluate_groups(fine_model,test_groups,model_stage="hidden_test",batch_size=64)
    print("FINAL HIDDEN-TEST metrics:"); display(hidden_overall)
    hidden_detail.to_csv(os.path.join(OUTPUT_ROOT,"hidden_test_group_metrics.csv"),index=False)
    hidden_by_domain.to_csv(os.path.join(OUTPUT_ROOT,"hidden_test_by_domain.csv"),index=False)
    hidden_by_reason.to_csv(os.path.join(OUTPUT_ROOT,"hidden_test_by_negative_reason.csv"),index=False)

Hidden test remains LOCKED. Set RUN_HIDDEN_TEST=True only after all train/validation model decisions are final.


## Use the trained model as MUFASA's retrieval second stage

A typical research-assistant pipeline becomes:

```text
vector/BM25 retrieval
        ↓
top 20–50 passages
        ↓
MiniLM cross-encoder reranker
        ↓
top 3–5 strongest evidence passages
        ↓
MUFASA LFM2.5 reasoning model
```

The reranker is intentionally separate from the generative LLM.

In [17]:
#@title 15. Rerank arbitrary research passages

@torch.inference_mode()
def rerank_candidates(
    query,
    candidates,
    top_k=5,
    model=fine_model,
):
    scores = score_query_docs(
        model,
        query,
        candidates,
        batch_size=64,
    )

    ranked = sorted(
        zip(candidates, scores),
        key=lambda x: x[1],
        reverse=True,
    )

    return ranked[:top_k]

# Example using one held-out validation group.
example_group = val_groups[0]

print("QUERY:")
print(example_group["query"])

print("\nRANKED CANDIDATES:")
for rank, (candidate, score) in enumerate(
    rerank_candidates(
        example_group["query"],
        example_group["docs"],
        top_k=len(example_group["docs"]),
    ),
    start=1,
):
    gold = (
        "POSITIVE"
        if candidate in [
            d for d, y in zip(
                example_group["docs"],
                example_group["labels"],
            )
            if y == 1
        ]
        else "NEGATIVE"
    )

    print(f"\n{rank}. score={score:.6f} | {gold}")
    print(candidate)

QUERY:
Which passage reports the seasonal dissolved-oxygen result rather than the BOD₅ result?

RANKED CANDIDATES:

1. score=0.049818 | POSITIVE
Concentrations for wet season(2.74 - 5.08mg/l) were higher than in the dry season (1.89-3.25mg/l).

2. score=0.008707 | NEGATIVE
The Biochemical oxygen demand (BOD₅) concentration for the study period varied from 1.00 to 2.14mg/l.


In [18]:
#@title 16. Measure model size and simple reranking throughput

model_bytes = sum(
    p.stat().st_size
    for p in Path(MODEL_DIR).rglob("*")
    if p.is_file()
)

print(
    "Saved model directory size:",
    round(model_bytes / 1024**2, 2),
    "MB",
)

# Simple local throughput benchmark using validation pairs.
bench_pairs = val_pairs[: min(512, len(val_pairs))]
bench_queries = [p["query"] for p in bench_pairs]
bench_docs = [p["doc"] for p in bench_pairs]

fine_model.to(DEVICE)
fine_model.eval()

if DEVICE == "cuda":
    torch.cuda.synchronize()

start = time.perf_counter()

with torch.inference_mode():
    for i in range(0, len(bench_pairs), 64):
        enc = tokenizer(
            bench_queries[i:i+64],
            bench_docs[i:i+64],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_tensors="pt",
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        _ = fine_model(**enc).logits

if DEVICE == "cuda":
    torch.cuda.synchronize()

elapsed = time.perf_counter() - start
pairs_per_second = len(bench_pairs) / elapsed

print("Benchmark pairs:", len(bench_pairs))
print("Elapsed seconds:", round(elapsed, 4))
print("Query-candidate pairs/sec:", round(pairs_per_second, 2))

Saved model directory size: 87.34 MB
Benchmark pairs: 512
Elapsed seconds: 0.2196
Query-candidate pairs/sec: 2331.61


## Optional: upload the trained reranker to Kaggle privately

The base MiniLM reranker is Apache-2.0 licensed. This cell stages the trained model files and evaluation metrics as a private Kaggle dataset.

It does **not** upload your paper JSON corpus.

In [19]:
#@title 17. Kaggle authentication

from google.colab import files

KAGGLE_DIR = Path("/root/.kaggle")
KAGGLE_JSON = KAGGLE_DIR / "kaggle.json"

if not KAGGLE_JSON.exists():
    print(
        "Upload kaggle.json from Kaggle -> "
        "Settings -> API -> Create New Token"
    )
    uploaded = files.upload()

    if not uploaded:
        raise RuntimeError("No credential file uploaded.")

    if "kaggle.json" in uploaded:
        credential_bytes = uploaded["kaggle.json"]
    else:
        first_name = next(iter(uploaded))
        credential_bytes = uploaded[first_name]

    KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
    with open(KAGGLE_JSON, "wb") as f:
        f.write(credential_bytes)

os.chmod(KAGGLE_JSON, 0o600)

auth = __import__("subprocess").run(
    ["kaggle", "datasets", "list", "-m", "-p", "1"],
    text=True,
    capture_output=True,
)

if auth.returncode != 0:
    print(auth.stdout)
    print(auth.stderr)
    raise RuntimeError("Kaggle authentication failed.")

print("Kaggle authentication looks good.")

Upload kaggle.json from Kaggle -> Settings -> API -> Create New Token


Saving kaggle(2).json to kaggle(2).json
Kaggle authentication looks good.


In [20]:
#@title 18. Stage model + metrics for private Kaggle upload

import subprocess

UPLOAD_DIR = Path("/content/kaggle_upload_mufasa_reranker")

if UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)

UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Copy trained HF model/tokenizer files.
for p in Path(MODEL_DIR).iterdir():
    if p.is_file():
        shutil.copy2(p, UPLOAD_DIR / p.name)

# Copy evaluation/training artifacts, but NOT paper JSON data.
for name in [
    "training_metrics.json",
    "raw_vs_finetuned_metrics.csv",
    "raw_base_group_metrics.csv",
    "fine_tuned_group_metrics.csv",
    "fine_tuned_by_domain.csv",
    "fine_tuned_by_negative_reason.csv",
]:
    src = Path(OUTPUT_ROOT) / name
    if src.exists():
        shutil.copy2(src, UPLOAD_DIR / name)

readme = f"""MUFASA MiniLM-L6 Research Evidence Reranker

Base model:
  {BASE_MODEL}

Task:
  Cross-encoder scientific evidence reranking.

Training supervision:
  tasks.training.payload.reranker from local paper JSON files.
  Each source reranker record provides a query, positive_quote,
  and hard_negative_quote.

Training:
  BCEWithLogitsLoss
  max_length={MAX_LENGTH}
  epochs={NUM_EPOCHS}
  train_batch_size={TRAIN_BATCH_SIZE}
  eval_batch_size={EVAL_BATCH_SIZE}
  learning_rate={LEARNING_RATE}
  weight_decay={WEIGHT_DECAY}
  seed={SEED}

Validation:
  Entire paper_id values held out to prevent same-paper leakage.

Base-model license:
  Apache-2.0

Source research papers/data may have their own licenses.
This Kaggle package does not include the source paper JSON corpus.
"""

with open(
    UPLOAD_DIR / "README.txt",
    "w",
    encoding="utf-8",
) as f:
    f.write(readme)

metadata = {
    "title": KAGGLE_DATASET_TITLE,
    "id": KAGGLE_DATASET_SLUG,
    "licenses": [{"name": "apache-2.0"}],
    "description": (
        "MUFASA MiniLM-L6 scientific research evidence cross-encoder "
        "reranker, fine-tuned from cross-encoder/ms-marco-MiniLM-L6-v2."
    ),
}

with open(
    UPLOAD_DIR / "dataset-metadata.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(metadata, f, indent=2)

print("Files staged:")
for p in sorted(UPLOAD_DIR.iterdir()):
    print(" ", p.name, f"({p.stat().st_size / 1e6:.2f} MB)")

print("\nTarget:", KAGGLE_DATASET_SLUG)
print("Visibility: PRIVATE")

Files staged:
  README.txt (0.00 MB)
  config.json (0.00 MB)
  dataset-metadata.json (0.00 MB)
  fine_tuned_by_domain.csv (0.00 MB)
  fine_tuned_by_negative_reason.csv (0.00 MB)
  fine_tuned_group_metrics.csv (0.22 MB)
  model.safetensors (90.87 MB)
  raw_base_group_metrics.csv (0.22 MB)
  raw_vs_finetuned_metrics.csv (0.00 MB)
  tokenizer.json (0.71 MB)
  tokenizer_config.json (0.00 MB)
  training_args.bin (0.01 MB)
  training_metrics.json (0.00 MB)

Target: williamalabi/mufasa-minilm-l6-research-reranker
Visibility: PRIVATE


In [21]:
#@title 19. Create private Kaggle dataset or push a new version

create_result = subprocess.run(
    [
        "kaggle", "datasets", "create",
        "-p", str(UPLOAD_DIR),
        "--dir-mode", "zip",
    ],
    text=True,
    capture_output=True,
)

if create_result.returncode == 0:
    print(create_result.stdout)
    print(
        "Created private dataset:",
        f"https://www.kaggle.com/datasets/{KAGGLE_DATASET_SLUG}",
    )
else:
    print("Create failed; trying a new dataset version.")
    print(create_result.stdout)
    print(create_result.stderr)

    version_result = subprocess.run(
        [
            "kaggle", "datasets", "version",
            "-p", str(UPLOAD_DIR),
            "-m",
            f"MUFASA reranker update "
            f"{time.strftime('%Y-%m-%d %H:%M:%S')}",
            "--dir-mode", "zip",
        ],
        text=True,
        capture_output=True,
    )

    print(version_result.stdout)

    if version_result.returncode != 0:
        print(version_result.stderr)
        raise RuntimeError(
            "Kaggle upload failed. Check credentials, slug, "
            "and metadata above."
        )

    print(
        "Pushed new private version:",
        f"https://www.kaggle.com/datasets/{KAGGLE_DATASET_SLUG}",
    )

Starting upload for file config.json
Upload successful: config.json (922B)
Starting upload for file README.txt
Upload successful: README.txt (718B)
Starting upload for file fine_tuned_by_domain.csv
Upload successful: fine_tuned_by_domain.csv (889B)
Starting upload for file tokenizer_config.json
Upload successful: tokenizer_config.json (443B)
Starting upload for file training_metrics.json
Upload successful: training_metrics.json (193B)
Starting upload for file tokenizer.json
Upload successful: tokenizer.json (695KB)
Starting upload for file fine_tuned_group_metrics.csv
Upload successful: fine_tuned_group_metrics.csv (219KB)
Starting upload for file model.safetensors
Upload successful: model.safetensors (87MB)
Starting upload for file raw_base_group_metrics.csv
Upload successful: raw_base_group_metrics.csv (215KB)
Starting upload for file raw_vs_finetuned_metrics.csv
Upload successful: raw_vs_finetuned_metrics.csv (697B)
Starting upload for file training_args.bin
Upload successful: train

## Notes

This notebook deliberately keeps the reference reranker's simple pointwise binary loss. Your source data already contains **hard negatives**, so there is no need to invent a separate contrastive objective to get a strong baseline.

The most meaningful validation number is **grouped top-1 accuracy**: for each held-out scientific query, did the positive evidence rank above its supplied hard negative?

MRR and nDCG are retained because they match the reference repository's evaluation family and become more informative if the dataset later contains groups with more than two candidate passages.